In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../src'))
if module_path not in sys.path:
    sys.path.append(module_path)

import vsopy.util
import vsopy.phot

OBJ_NAME='SZ_Lyn'
SESSION_TAG='2026/20260123'
IMAGE_ROOT = '/srv/public/img'
WORK_ROOT = '/srv/public'

session = vsopy.util.Session(tag=SESSION_TAG, name=OBJ_NAME)
layout = vsopy.util.WorkLayout(WORK_ROOT)
session_layout = layout.get_session(session)
settings = vsopy.util.Settings(session_layout.settings_file_path)

Traditional transformation approach

Let we have photometry data for a set of standard stas measured in two bands,
say $B$ and $V$.  Using capital letters for standard magnitudes and small
letters for instumental magnitudes, we're first correct for atmospheric 
extiction, finding exoatmospheric magnitues $b_0$ and $v_0$:

$$
\begin{gather}
v = v_0 + k_v X \\
b - v = (b_0-v_0) + k_{bv} X
\end{gather}
$$

where $X$ is the airmass for the observation.  $(b_0-v_0)$ is the instrumental 
color intex of the star.

Extiction coefficients $k_v$ and $k_{bv}$ typically depend on the color of the star:

$$
\begin{gather}
\tag{2}
k_v = k^{'}_v + k^{''}_v (b_0-v_0) \\
k_{bv} = k^{'}_{bv} + k^{''}_{bv} (b_0-v_0)
\end{gather}
$$

it gives the following expressions for exoatmospheric magnitudes:

$$ 
\begin{gather}
v_0 = v - k^{'}_v X - k^{''}_v (b_0-v_0) X \\
b_0-v_0 = \frac{(b-v)-k^{'}_{bv} X}{1 +k^{''}_{bv} X}
\end{gather}
$$

Now the color transformation proper. We assume that exoatmopheric instrumental 
magnitude and color index should be transformed by polynomial depending on  the
standard color index:

$$
\begin{gather}
v_0 = V + \sum_{i=0}^{n} \alpha_i (B-V)^i \\
b_0-v_0 = \sum_{i=0}^{n} \beta_i (B-V)^i
\end{gather}
$$

We assume that that transform is mostly linear, so second-order coefficients
should be small.  It worth calculating them to verify the linearity of the transform.

We don't want to make separate obsevations to determine extiction and transform
coefficients.  Insteed we're just taking all obervations of the ensemble over the 
night and use regression to find all coefficients at once.

$$
$$



In [ ]:
provider = vsopy.phot.BatchDataProvider(session_layout)
provider.images_.rename_column('filter', 'band')
provider.measured_.rename_column('M', 'm')

In [ ]:
from astropy.table import join
import numpy as np
data = join(join(provider.images_["image_id", "band", "airmass"],
                 provider.measured_["image_id", "auid", "m"],
                 keys=["image_id"]), provider.sequence_, keys=["auid", "band"])
data.colnames,

In [ ]:
dataB = data[data["band"] == "B"]
dataV = data[data["band"] == "V"]
dataB

In [ ]:
dataCI =join(join(join(dataB,
                        provider.batch_images_["batch_id", "image_id"],
                        keys=["image_id"])["m", "M", "batch_id", "auid"],
                  join(dataV,
                        provider.batch_images_["batch_id", "image_id"],
                        keys=["image_id"])["m", "M", "batch_id", "auid"],
                  keys=["batch_id", "auid"],
                  table_names=["B", "V"]),
              provider.batches_["batch_id", "airmass"],
              keys=["batch_id"])

dataCI["CI"] = dataCI["M_B"]["mag"] - dataCI["M_V"]["mag"]
dataCI["ci"] = dataCI["m_B"]["mag"] - dataCI["m_V"]["mag"]

datavV = dataCI
dataCI = dataCI[["CI", "ci", "airmass", "auid"]]
datavV["vV"] = datavV["m_V"]["mag"] - datavV["M_V"]["mag"]
datavV = datavV[["vV", "CI", "airmass", "auid"]]
#dataCI
#provider.batches_["batch_id", "airmass"]

In [ ]:
import numpy as np

X=np.vstack([np.ones(len(dataCI)),
           dataCI["airmass"],
           dataCI["CI"].value,
           dataCI["airmass"] * dataCI["CI"].value#,
        #    dataCI["CI"].value*dataCI["CI"].value,
        #    dataCI["airmass"] * dataCI["airmass"].value
        ]).T
np.set_printoptions(precision=2, suppress=False)
XTX = X.T @ X
XTXI = np.linalg.inv(XTX)
XX = XTXI @ X.T
np.diag(XTXI)
#XX @ s, np.diag(XX)

In [ ]:
a = XX @ datavV["vV"].value
a

In [ ]:
s2v = np.sum((X @ a - datavV["vV"].value)**2) / (len(datavV) - len(a))
sigmaa=np.diag(XTXI) * s2v
np.set_printoptions(precision=4, suppress=False)
sigmaa

In [ ]:
from uncertainties import unumpy
a_u = unumpy.uarray(a, np.sqrt(sigmaa))
k1v = (-a_u[0]*a_u[3] + a_u[1]*a_u[2])/a_u[2]
k2v = a_u[3]/a_u[2]
alpha1 = a_u[0]
alpha2 = a_u[2]
k1v, k2v, alpha1, alpha2

In [ ]:
b = XX @ dataCI["ci"].value
b

In [ ]:
s2bv = np.sum((X @ b - dataCI["ci"].value)**2) / (len(dataCI) - len(b))
sigmab=np.diag(XTXI) * s2bv
np.set_printoptions(precision=4, suppress=False)
sigmab

In [ ]:
b_u = unumpy.uarray(b, np.sqrt(sigmaa))
k1bv = (-b_u[0]*b_u[3] + b_u[1]*b_u[2])/b_u[2]
k2bv = b_u[3]/b_u[2]
beta1 = b_u[0]
beta2 = b_u[2]
k1bv, k2bv, beta1, beta2

In [ ]:
rv = datavV["vV", "auid"]
rv["r"] = np.sqrt((X @ a - rv["vV"].value)**2)
rv["auid", "r"].group_by("auid").groups.aggregate(np.mean)